In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import requests

In [2]:
# load.env-file
load_dotenv()

schema = "sql_cities"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

# APIs
x_rapid_api_key = os.getenv("X_RapidAPI_Key")

In [3]:
url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.31","lon":"13.24","radiusKm":"50","limit":"50","withFlightInfoOnly":"true"}

headers = {
    "Accept": "application/json, application/xml",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'message': 'You are not subscribed to this API.'}


In [4]:
code = "EDDB"
tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

time_windows = [
    (f"{tomorrow}T00:00", f"{tomorrow}T11:59"),
    (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
]

headers = {
    "Accept": "application/json",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

params = {
    "direction": "Arrival",
    "withLeg": "false",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false",
    "withLocation": "true"
}

all_flights = []

# requesting both timeframes and collecting all flights
for fromLocal, toLocal in time_windows:
    url = (
        f"https://aerodatabox.p.rapidapi.com/flights/"
        f"airports/icao/{code}/{fromLocal}/{toLocal}"
    )

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    result = response.json()
    arrivals = (
        result if isinstance(result, list)
        else result.get("arrivals", [])
    )

  #  print(f"{fromLocal} bis {toLocal}: {len(arrivals)} flights")
    all_flights.extend(arrivals)

# creation of df after both requests
cities_flight_data_df = pd.json_normalize(all_flights)
# sorting according scheduled time of arrival
time_column = "movement.scheduledTime.utc"

if time_column in cities_flight_data_df.columns:
    cities_flight_data_df[time_column] = pd.to_datetime(
        cities_flight_data_df[time_column],
        utc=True,
        errors="coerce"
    )

    cities_flight_data_df = cities_flight_data_df.sort_values(
        by=time_column,
        ascending=True,
        na_position="last"
    )

#print(f"\nArrivals at airport {code} on {tomorrow}:")
#print(f"Sum of flights: {len(flight_data_df)}")

columns_to_show = [
    col for col in [
        "number",
        "airline.name",
        #"movement.scheduledTime.local",
        "movement.scheduledTime.utc",
        "status"
    ]
    if col in cities_flight_data_df.columns
]

#print(cities_flight_data_df[columns_to_show].to_string(index=False))

cities_flight_data_df = cities_flight_data_df[columns_to_show].copy()

cities_flight_data_df = cities_flight_data_df.rename(columns={
    "number": "flight_number",
    "airline.name": "airline_cs",
    "movement.scheduledTime.utc": "scheduled_time_utc"
})

cities_flight_data_df

HTTPError: 429 Client Error: Too Many Requests for url: https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/2026-10-24T00:00/2026-10-24T11:59?direction=Arrival&withLeg=false&withCancelled=false&withCodeshared=true&withCargo=false&withPrivate=false&withLocation=true

In [ ]:
cities_flight_data_df.to_sql('cities_flight_data',
                  if_exists='append',
                  con=connection_string,
                  index=False)